# Notebook 04 — Feature, ZigZag Timing, and Leakage Audit

## Purpose

This notebook freezes the model-input schema and converts the ZigZag design
intent into an explicitly confirmation-gated, prefix-invariant primary long-side
generator.

## Non-negotiable controls

- Feature approval uses **train only**.
- Unseen-test outcomes are not read for feature or threshold selection.
- Label, event-end, censoring, barrier, and legacy future-derived fields are never model inputs.
- Legacy `zigzag_up_new_2` and `zigzag_down_new_2` are audited but are **not**
  used directly as model features or as the primary event filter.
- The ZigZag state is reconstructed from adjusted price history and becomes
  available only at the pivot confirmation observation.
- The **15%** candidate threshold is the pre-registered primary rule.
- **10%** and **20%** are train-only sensitivity diagnostics.
- The confirmed ZigZag rule supplies `side = long`; within those candidate
  events, the frozen Triple-Barrier `label` is the take/skip meta-label.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate the repository and import project modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Run this notebook from inside the project."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.features.feature_schema import (
    audit_prohibited_feature_names,
    build_feature_role_table,
    finalize_feature_approval,
)
from src.features.leakage_checks import (
    ConfirmedZigZagConfig,
    build_candidate_long_mask,
    build_confirmation_gated_zigzag_state,
    legacy_zigzag_source_logic_audit,
    prefix_invariance_audit,
)
from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load frozen Stage 03 policy and Stage 04 feature policy

In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
columns_config = load_yaml(REPOSITORY_ROOT / "configs" / "columns.yaml")
labeling_config = load_yaml(REPOSITORY_ROOT / "configs" / "labeling.yaml")
validation_config = load_yaml(REPOSITORY_ROOT / "configs" / "validation.yaml")

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"
CANDIDATE_TRAIN_PATH = DATA_PATHS["candidate_data"] / "train"
CANDIDATE_TRAIN_PATH.mkdir(parents=True, exist_ok=True)

DATE_COLUMN = "dEven"
OUTPUT_ENCODING = "utf-8-sig"

feature_policy = columns_config["feature_policy"]
zigzag_policy = columns_config["zigzag_audit"]
candidate_policy = columns_config["candidate_event_filter"]

assert labeling_config["status"] == "stage_03_frozen"
assert labeling_config["frozen_for_experiment"] is True
assert labeling_config["selected_scenario"] == "symmetric_15_15"
assert labeling_config["zigzag_policy"]["used_to_construct_label"] is False
assert labeling_config["zigzag_policy"]["formal_row_level_audit_stage"] == "Notebook 04"

PRIMARY_THRESHOLD = float(candidate_policy["primary_threshold_fraction"])
SENSITIVITY_THRESHOLDS = [
    float(value)
    for value in candidate_policy["sensitivity_threshold_fractions"]
]
ALL_THRESHOLDS = sorted(set([PRIMARY_THRESHOLD, *SENSITIVITY_THRESHOLDS]))

assert candidate_policy["selection_scope"] == "train_only"
assert candidate_policy["automatic_threshold_selection"] is False
assert np.isclose(
    PRIMARY_THRESHOLD,
    float(labeling_config["zigzag_policy"]["intended_filter_threshold_fraction"]),
)
assert np.isclose(PRIMARY_THRESHOLD, 0.15)

zigzag_config = ConfirmedZigZagConfig(
    depth=int(zigzag_policy["source_depth"]),
    deviation_percent=float(zigzag_policy["source_deviation_percent"]),
)

print("Labeled train input:", LABELED_TRAIN_PATH)
print("Candidate train output:", CANDIDATE_TRAIN_PATH)
print("Primary filter threshold:", PRIMARY_THRESHOLD)
print("Sensitivity thresholds:", SENSITIVITY_THRESHOLDS)
print("Unseen-test used for Stage 04 approval: False")


Labeled train input: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\labeled\train
Candidate train output: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\candidates\train
Primary filter threshold: 0.15
Sensitivity thresholds: [0.1, 0.2]
Unseen-test used for Stage 04 approval: False


## 3. Validate the frozen universe and Stage 03 labeled train files

In [4]:
frozen_universe_path = RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
if not frozen_universe_path.exists():
    raise FileNotFoundError(
        "Frozen universe manifest is missing. Run and freeze Notebook 02 first."
    )

frozen_universe_df = pd.read_csv(frozen_universe_path, low_memory=False)
frozen_symbols = sorted(frozen_universe_df["symbol"].astype(str).tolist())
frozen_symbol_set = set(frozen_symbols)

train_files = sorted(LABELED_TRAIN_PATH.glob("*_train_labeled.csv"))

def symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected file name: {path.name}")
    return path.name[:-len(suffix)]

train_file_map = {
    symbol_from_path(path): path
    for path in train_files
}

assert set(train_file_map) == frozen_symbol_set
assert len(train_files) == len(frozen_symbols)

print("Frozen universe size:", len(frozen_symbols))
print("Labeled train files:", len(train_files))


Frozen universe size: 499
Labeled train files: 499


## 4. Structural feature-role and prohibited-name audit

In [5]:
candidate_features = list(columns_config["candidate_features"])

role_table_df = build_feature_role_table(columns_config)
prohibited_name_audit_df = audit_prohibited_feature_names(
    candidate_features,
    list(feature_policy["prohibited_feature_name_patterns"]),
)

assert role_table_df["role_membership_count"].eq(1).all()
assert not prohibited_name_audit_df["prohibited_name_hit"].any()

role_table_path = RESULT_PATHS["audits"] / "04_feature_role_audit.csv"
prohibited_path = RESULT_PATHS["audits"] / "04_prohibited_name_audit.csv"

role_table_df.to_csv(role_table_path, index=False, encoding=OUTPUT_ENCODING)
prohibited_name_audit_df.to_csv(
    prohibited_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)

display(role_table_df)


,feature,structural_role,structurally_approved,structural_reason,role_membership_count
0,adj_open,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
1,adj_high,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
2,adj_low,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
3,adj_last_price,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
4,zigzag_up_new_2,legacy_zigzag_rejected,False,Legacy new_2 ZigZag columns are replaced by a ...,1
5,zigzag_down_new_2,legacy_zigzag_rejected,False,Legacy new_2 ZigZag columns are replaced by a ...,1
6,EMA_3,model_candidate,True,Eligible for train-only data-quality approval.,1
7,EMA_5,model_candidate,True,Eligible for train-only data-quality approval.,1
8,EMA_8,model_candidate,True,Eligible for train-only data-quality approval.,1
9,EMA_10,model_candidate,True,Eligible for train-only data-quality approval.,1


## 5. Train-only feature quality audit

In [6]:
quality_accumulator = {
    feature: {
        "total_rows": 0,
        "missing_values": 0,
        "finite_values": 0,
        "nonfinite_values": 0,
        "global_min": np.inf,
        "global_max": -np.inf,
        "symbols_present": 0,
    }
    for feature in candidate_features
}

schema_error_rows = []
started = time.perf_counter()

required_columns = {
    DATE_COLUMN,
    "eligible_for_modeling",
    "label",
    "barrier_touched",
    "holding_period_observations",
    "event_return",
    "adj_high",
    "adj_low",
    "adj_last_price",
    *candidate_features,
}

for file_number, (symbol, path) in enumerate(
    sorted(train_file_map.items()),
    start=1,
):
    try:
        frame = pd.read_csv(path, low_memory=False)

        missing_columns = sorted(required_columns - set(frame.columns))
        if missing_columns:
            raise KeyError(
                f"{path.name}: missing required columns {missing_columns}"
            )

        eligible = frame["eligible_for_modeling"].fillna(False).astype(bool)
        eligible_frame = frame.loc[eligible]

        for feature in candidate_features:
            values = pd.to_numeric(
                eligible_frame[feature],
                errors="coerce",
            )
            array = values.to_numpy(dtype=float)
            finite = np.isfinite(array)

            target = quality_accumulator[feature]
            target["total_rows"] += int(len(array))
            target["missing_values"] += int(values.isna().sum())
            target["finite_values"] += int(finite.sum())
            target["nonfinite_values"] += int(
                np.isinf(array).sum()
            )
            target["symbols_present"] += 1

            if finite.any():
                target["global_min"] = min(
                    target["global_min"],
                    float(np.min(array[finite])),
                )
                target["global_max"] = max(
                    target["global_max"],
                    float(np.max(array[finite])),
                )

    except Exception as exc:
        schema_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 25 == 0
        or file_number == len(train_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[feature quality] "
            f"[{file_number:>4}/{len(train_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(schema_error_rows)}"
        )

quality_rows = []

for feature, values in quality_accumulator.items():
    total_rows = int(values["total_rows"])
    finite_values = int(values["finite_values"])
    global_min = (
        float(values["global_min"])
        if np.isfinite(values["global_min"])
        else np.nan
    )
    global_max = (
        float(values["global_max"])
        if np.isfinite(values["global_max"])
        else np.nan
    )

    quality_rows.append(
        {
            "feature": feature,
            **values,
            "global_min": global_min,
            "global_max": global_max,
            "missing_fraction": (
                int(values["missing_values"]) / total_rows
                if total_rows
                else np.nan
            ),
            "nonfinite_fraction": (
                int(values["nonfinite_values"]) / total_rows
                if total_rows
                else np.nan
            ),
            "is_constant_on_finite_train_values": bool(
                finite_values > 0
                and np.isclose(global_min, global_max)
            ),
        }
    )

feature_quality_df = pd.DataFrame(quality_rows)
schema_errors_df = pd.DataFrame(
    schema_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

quality_config = feature_policy["data_quality"]

feature_approval_df = finalize_feature_approval(
    role_table_df,
    feature_quality_df,
    maximum_missing_fraction=float(
        quality_config["maximum_missing_fraction_for_approval"]
    ),
    reject_all_missing=bool(quality_config["reject_all_missing"]),
    reject_constant=bool(quality_config["reject_constant"]),
)

approved_model_features = feature_approval_df.loc[
    feature_approval_df["approved_for_modeling"],
    "feature",
].astype(str).tolist()

feature_quality_df.to_csv(
    RESULT_PATHS["audits"] / "04_feature_quality_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
feature_approval_df.to_csv(
    RESULT_PATHS["audits"] / "04_feature_approval_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
schema_errors_df.to_csv(
    RESULT_PATHS["audits"] / "04_feature_audit_errors.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

approved_features_df = pd.DataFrame(
    {
        "feature_order": range(1, len(approved_model_features) + 1),
        "feature": approved_model_features,
    }
)
approved_features_df.to_csv(
    RESULT_PATHS["manifests"] / "04_approved_model_features.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

print("Approved model features:", len(approved_model_features))
display(
    feature_approval_df[
        [
            "feature",
            "structural_role",
            "missing_fraction",
            "is_constant_on_finite_train_values",
            "approved_for_modeling",
            "approval_reason",
        ]
    ]
)


[feature quality] [   1/499] elapsed=0.2s errors=0
[feature quality] [  25/499] elapsed=2.5s errors=0
[feature quality] [  50/499] elapsed=4.6s errors=0
[feature quality] [  75/499] elapsed=6.6s errors=0
[feature quality] [ 100/499] elapsed=8.5s errors=0
[feature quality] [ 125/499] elapsed=10.7s errors=0
[feature quality] [ 150/499] elapsed=12.5s errors=0
[feature quality] [ 175/499] elapsed=14.5s errors=0
[feature quality] [ 200/499] elapsed=16.3s errors=0
[feature quality] [ 225/499] elapsed=18.1s errors=0
[feature quality] [ 250/499] elapsed=20.4s errors=0
[feature quality] [ 275/499] elapsed=22.3s errors=0
[feature quality] [ 300/499] elapsed=24.2s errors=0
[feature quality] [ 325/499] elapsed=26.3s errors=0
[feature quality] [ 350/499] elapsed=28.4s errors=0
[feature quality] [ 375/499] elapsed=30.3s errors=0
[feature quality] [ 400/499] elapsed=32.2s errors=0
[feature quality] [ 425/499] elapsed=34.0s errors=0
[feature quality] [ 450/499] elapsed=35.9s errors=0
[feature quality]

,feature,structural_role,missing_fraction,is_constant_on_finite_train_values,approved_for_modeling,approval_reason
0,adj_open,context_only,0.000000,False,False,context_only
1,adj_high,context_only,0.000000,False,False,context_only
2,adj_low,context_only,0.000000,False,False,context_only
3,adj_last_price,context_only,0.000000,False,False,context_only
4,zigzag_up_new_2,legacy_zigzag_rejected,0.012292,False,False,legacy_zigzag_rejected
5,zigzag_down_new_2,legacy_zigzag_rejected,0.012292,False,False,legacy_zigzag_rejected
6,EMA_3,model_candidate,0.001543,False,True,approved
7,EMA_5,model_candidate,0.003087,False,True,approved
8,EMA_8,model_candidate,0.005402,False,True,approved
9,EMA_10,model_candidate,0.006945,False,True,approved


## 6. Audit the supplied legacy ZigZag timing logic

The supplied collection code contains explicit pivot confirmation indices. The
legacy `new_2` distance loop, however, scans the pivot marker and skips the first
pivot rather than directly requiring the confirmation marker.

Stage 04 preserves the original **confirmation intent** and the original
`depth=10`, `deviation=15%` parameters, but replaces the skip-first heuristic
with a direct confirmation-time gate.


In [7]:
legacy_zigzag_logic_df = legacy_zigzag_source_logic_audit()

legacy_zigzag_logic_df.to_csv(
    RESULT_PATHS["audits"] / "04_legacy_zigzag_source_logic_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

display(legacy_zigzag_logic_df)


,check,observed,risk,stage_04_action
0,confirmation_index_is_computed,True,False,retain design intent
1,confirmation_markers_are_created,True,False,retain design intent
2,legacy_new2_loop_scans_pivot_marker_zj,True,True,do not use legacy new_2 columns directly
3,legacy_new2_loop_explicitly_requires_confirmat...,False,True,reconstruct confirmation-gated state
4,legacy_new2_loop_skips_first_pivot,True,True,replace heuristic with confirmation timestamp ...
5,stage04_state_requires_confirmation_index_at_o...,True,False,required
6,stage04_state_is_prefix_invariance_audited,True,False,required


## 7. Reconstruct confirmation-gated ZigZag state and audit prefix invariance

Prefix invariance means that the state recorded for date `t` is identical whether
we calculate on data ending at `t` or on the full train history. This directly
tests the absence of historical repainting in the Stage 04 reconstruction.


In [8]:
rng = np.random.default_rng(1729)

sample_symbol_count = min(12, len(frozen_symbols))
prefix_positions_per_symbol = 12

sampled_symbols = sorted(
    rng.choice(
        np.asarray(frozen_symbols, dtype=object),
        size=sample_symbol_count,
        replace=False,
    ).tolist()
)

prefix_audit_rows = []

for symbol in sampled_symbols:
    path = train_file_map[symbol]
    frame = pd.read_csv(
        path,
        low_memory=False,
        usecols=[
            DATE_COLUMN,
            "adj_high",
            "adj_low",
            "adj_last_price",
        ],
    )
    frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN], errors="coerce")
    frame = frame.sort_values(DATE_COLUMN, kind="stable").reset_index(drop=True)

    minimum_position = max(
        64,
        zigzag_config.depth * 4,
    )

    if len(frame) <= minimum_position:
        positions = []
    else:
        candidate_positions = np.arange(
            minimum_position,
            len(frame),
            dtype=int,
        )
        sample_size = min(
            prefix_positions_per_symbol,
            len(candidate_positions),
        )
        positions = sorted(
            rng.choice(
                candidate_positions,
                size=sample_size,
                replace=False,
            ).tolist()
        )

    symbol_audit = prefix_invariance_audit(
        frame,
        config=zigzag_config,
        positions=positions,
        date_column=DATE_COLUMN,
    )
    symbol_audit.insert(0, "symbol", symbol)
    prefix_audit_rows.append(symbol_audit)

prefix_invariance_df = pd.concat(
    prefix_audit_rows,
    ignore_index=True,
) if prefix_audit_rows else pd.DataFrame(
    columns=[
        "symbol",
        "position",
        "event_date",
        "prefix_invariant",
        "mismatch_columns",
    ]
)

prefix_invariance_df.to_csv(
    RESULT_PATHS["audits"] / "04_zigzag_prefix_invariance_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

prefix_mismatches = int(
    (~prefix_invariance_df["prefix_invariant"].astype(bool)).sum()
)

print("Prefix checks:", len(prefix_invariance_df))
print("Prefix mismatches:", prefix_mismatches)
display(prefix_invariance_df.head(20))


Prefix checks: 144
Prefix mismatches: 0


,symbol,position,event_date,prefix_invariant,mismatch_columns
0,بالبر,255,2015-11-02,True,
1,بالبر,290,2016-01-12,True,
2,بالبر,345,2016-04-06,True,
3,بالبر,434,2016-08-30,True,
4,بالبر,500,2016-12-25,True,
5,بالبر,575,2017-05-14,True,
6,بالبر,587,2017-05-30,True,
7,بالبر,628,2017-08-01,True,
8,بالبر,782,2018-03-27,True,
9,بالبر,1028,2019-04-22,True,


## 8. Train-only candidate-filter sensitivity and primary candidate generation

The 15% threshold is **not selected from these results**. It was already recorded
as the intended Stage 03 filter threshold.

The 10% and 20% rules are reported only to show how event density and label
composition change on train.


In [9]:
for old_file in CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv"):
    old_file.unlink()

threshold_accumulators = {
    threshold: defaultdict(float)
    for threshold in ALL_THRESHOLDS
}
threshold_year_accumulators = {
    threshold: defaultdict(
        lambda: defaultdict(float)
    )
    for threshold in ALL_THRESHOLDS
}

candidate_audit_rows = []
zigzag_state_audit_rows = []
candidate_write_errors = []

started = time.perf_counter()

for file_number, (symbol, path) in enumerate(
    sorted(train_file_map.items()),
    start=1,
):
    try:
        frame = pd.read_csv(path, low_memory=False)
        frame[DATE_COLUMN] = pd.to_datetime(
            frame[DATE_COLUMN],
            errors="coerce",
        )
        frame = frame.sort_values(
            DATE_COLUMN,
            kind="stable",
        ).reset_index(drop=True)

        state = build_confirmation_gated_zigzag_state(
            frame[
                [
                    DATE_COLUMN,
                    "adj_high",
                    "adj_low",
                    "adj_last_price",
                ]
            ],
            config=zigzag_config,
            date_column=DATE_COLUMN,
        )

        assert len(state) == len(frame)
        assert state[DATE_COLUMN].equals(frame[DATE_COLUMN])

        high_confirmation_after_row = (
            state["confirmed_zigzag_high_confirmation_date"].notna()
            & (
                pd.to_datetime(
                    state["confirmed_zigzag_high_confirmation_date"],
                    errors="coerce",
                )
                > frame[DATE_COLUMN]
            )
        )
        low_confirmation_after_row = (
            state["confirmed_zigzag_low_confirmation_date"].notna()
            & (
                pd.to_datetime(
                    state["confirmed_zigzag_low_confirmation_date"],
                    errors="coerce",
                )
                > frame[DATE_COLUMN]
            )
        )

        zigzag_state_audit_rows.append(
            {
                "symbol": symbol,
                "rows": len(frame),
                "state_available_rows": int(
                    state["confirmed_zigzag_state_available"].sum()
                ),
                "high_confirmation_after_event_rows": int(
                    high_confirmation_after_row.sum()
                ),
                "low_confirmation_after_event_rows": int(
                    low_confirmation_after_row.sum()
                ),
                "legacy_up_finite_rows": int(
                    np.isfinite(
                        pd.to_numeric(
                            frame["zigzag_up_new_2"],
                            errors="coerce",
                        ).to_numpy(dtype=float)
                    ).sum()
                ),
                "legacy_down_finite_rows": int(
                    np.isfinite(
                        pd.to_numeric(
                            frame["zigzag_down_new_2"],
                            errors="coerce",
                        ).to_numpy(dtype=float)
                    ).sum()
                ),
            }
        )

        eligible = frame["eligible_for_modeling"].fillna(False).astype(bool)

        for threshold in ALL_THRESHOLDS:
            candidate_mask = build_candidate_long_mask(
                state,
                eligible_for_modeling=eligible,
                threshold_fraction=threshold,
            )

            selected = frame.loc[candidate_mask]
            accumulator = threshold_accumulators[threshold]

            accumulator["rows"] += len(frame)
            accumulator["eligible_events"] += int(eligible.sum())
            accumulator["candidate_events"] += int(candidate_mask.sum())
            accumulator["positive_labels"] += int(
                selected["label"].eq(1).sum()
            )
            accumulator["negative_labels"] += int(
                selected["label"].eq(0).sum()
            )
            accumulator["upper_events"] += int(
                selected["barrier_touched"].eq("upper").sum()
            )
            accumulator["lower_events"] += int(
                selected["barrier_touched"].eq("lower").sum()
            )
            accumulator["vertical_events"] += int(
                selected["barrier_touched"].eq("vertical").sum()
            )
            accumulator["event_return_sum"] += float(
                pd.to_numeric(
                    selected["event_return"],
                    errors="coerce",
                ).sum()
            )
            accumulator["event_return_count"] += int(
                pd.to_numeric(
                    selected["event_return"],
                    errors="coerce",
                ).notna().sum()
            )

            if len(selected):
                selected_years = pd.to_datetime(
                    selected[DATE_COLUMN],
                    errors="coerce",
                ).dt.year

                for year, year_index in selected_years.groupby(
                    selected_years
                ).groups.items():
                    if pd.isna(year):
                        continue
                    year_selected = selected.loc[year_index]
                    year_target = threshold_year_accumulators[
                        threshold
                    ][int(year)]
                    year_target["candidate_events"] += len(year_selected)
                    year_target["positive_labels"] += int(
                        year_selected["label"].eq(1).sum()
                    )
                    year_target["negative_labels"] += int(
                        year_selected["label"].eq(0).sum()
                    )

        primary_mask = build_candidate_long_mask(
            state,
            eligible_for_modeling=eligible,
            threshold_fraction=PRIMARY_THRESHOLD,
        )

        primary_candidates = frame.loc[primary_mask].copy()
        primary_state = state.loc[primary_mask].reset_index(drop=True)

        state_columns_to_attach = [
            "confirmed_zigzag_high_price",
            "confirmed_zigzag_low_price",
            "confirmed_zigzag_high_pivot_date",
            "confirmed_zigzag_low_pivot_date",
            "confirmed_zigzag_high_confirmation_date",
            "confirmed_zigzag_low_confirmation_date",
            "distance_above_confirmed_low_fraction",
            "signed_distance_from_confirmed_high_fraction",
            "distance_below_confirmed_high_fraction",
            "confirmed_zigzag_state_available",
        ]

        primary_candidates = primary_candidates.reset_index(drop=True)
        primary_candidates[state_columns_to_attach] = primary_state[
            state_columns_to_attach
        ]

        primary_candidates["primary_side"] = 1
        primary_candidates["meta_label"] = pd.to_numeric(
            primary_candidates["label"],
            errors="coerce",
        ).astype("Int64")
        primary_candidates["candidate_filter_threshold_fraction"] = (
            PRIMARY_THRESHOLD
        )

        output_path = (
            CANDIDATE_TRAIN_PATH
            / f"{symbol}_train_candidates.csv"
        )
        primary_candidates.to_csv(
            output_path,
            index=False,
            encoding=OUTPUT_ENCODING,
        )

        candidate_dates = pd.to_datetime(
            primary_candidates[DATE_COLUMN],
            errors="coerce",
        ).sort_values()
        calendar_gaps = candidate_dates.diff().dt.days.dropna()

        candidate_audit_rows.append(
            {
                "symbol": symbol,
                "eligible_train_events": int(eligible.sum()),
                "primary_candidate_events": int(primary_mask.sum()),
                "candidate_fraction_of_eligible": (
                    float(primary_mask.sum() / eligible.sum())
                    if int(eligible.sum())
                    else np.nan
                ),
                "positive_meta_labels": int(
                    primary_candidates["meta_label"].eq(1).sum()
                ),
                "negative_meta_labels": int(
                    primary_candidates["meta_label"].eq(0).sum()
                ),
                "positive_meta_label_fraction": (
                    float(primary_candidates["meta_label"].eq(1).mean())
                    if len(primary_candidates)
                    else np.nan
                ),
                "median_calendar_gap_between_candidates": (
                    float(calendar_gaps.median())
                    if len(calendar_gaps)
                    else np.nan
                ),
                "first_candidate_date": (
                    candidate_dates.min()
                    if len(candidate_dates)
                    else pd.NaT
                ),
                "last_candidate_date": (
                    candidate_dates.max()
                    if len(candidate_dates)
                    else pd.NaT
                ),
                "output_rows": len(primary_candidates),
                "output_file": output_path.name,
            }
        )

    except Exception as exc:
        candidate_write_errors.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 25 == 0
        or file_number == len(train_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[ZigZag/candidate] "
            f"[{file_number:>4}/{len(train_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(candidate_write_errors)}"
        )

threshold_rows = []

for threshold in ALL_THRESHOLDS:
    values = threshold_accumulators[threshold]
    candidate_events = int(values["candidate_events"])
    eligible_events = int(values["eligible_events"])

    threshold_rows.append(
        {
            "threshold_fraction": threshold,
            "is_primary_pre_registered_threshold": np.isclose(
                threshold,
                PRIMARY_THRESHOLD,
            ),
            "comparison_scope": "train_only",
            "automatic_threshold_selection_used": False,
            **{
                key: int(value)
                if key not in {
                    "event_return_sum",
                }
                else float(value)
                for key, value in values.items()
                if key != "event_return_count"
            },
            "candidate_fraction_of_eligible": (
                candidate_events / eligible_events
                if eligible_events
                else np.nan
            ),
            "positive_meta_label_fraction": (
                values["positive_labels"] / candidate_events
                if candidate_events
                else np.nan
            ),
            "mean_event_return": (
                values["event_return_sum"]
                / values["event_return_count"]
                if values["event_return_count"]
                else np.nan
            ),
        }
    )

threshold_comparison_df = pd.DataFrame(threshold_rows)

threshold_year_rows = []
for threshold in ALL_THRESHOLDS:
    for year, values in sorted(
        threshold_year_accumulators[threshold].items()
    ):
        candidate_events = int(values["candidate_events"])
        threshold_year_rows.append(
            {
                "threshold_fraction": threshold,
                "calendar_year": int(year),
                **{
                    key: int(value)
                    for key, value in values.items()
                },
                "positive_meta_label_fraction": (
                    values["positive_labels"] / candidate_events
                    if candidate_events
                    else np.nan
                ),
            }
        )

threshold_by_year_df = pd.DataFrame(threshold_year_rows)
candidate_audit_df = pd.DataFrame(candidate_audit_rows)
zigzag_state_audit_df = pd.DataFrame(zigzag_state_audit_rows)
candidate_errors_df = pd.DataFrame(
    candidate_write_errors,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

threshold_comparison_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_filter_threshold_comparison.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
threshold_by_year_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_filter_by_year.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
candidate_audit_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_event_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
zigzag_state_audit_df.to_csv(
    RESULT_PATHS["audits"] / "04_confirmation_gated_zigzag_state_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_generation_errors.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

display(threshold_comparison_df)


[ZigZag/candidate] [   1/499] elapsed=0.4s errors=0
[ZigZag/candidate] [  25/499] elapsed=4.5s errors=0
[ZigZag/candidate] [  50/499] elapsed=8.6s errors=0
[ZigZag/candidate] [  75/499] elapsed=12.7s errors=0
[ZigZag/candidate] [ 100/499] elapsed=16.7s errors=0
[ZigZag/candidate] [ 125/499] elapsed=20.8s errors=0
[ZigZag/candidate] [ 150/499] elapsed=25.0s errors=0
[ZigZag/candidate] [ 175/499] elapsed=28.9s errors=0
[ZigZag/candidate] [ 200/499] elapsed=32.4s errors=0
[ZigZag/candidate] [ 225/499] elapsed=36.2s errors=0
[ZigZag/candidate] [ 250/499] elapsed=40.1s errors=0
[ZigZag/candidate] [ 275/499] elapsed=43.8s errors=0
[ZigZag/candidate] [ 300/499] elapsed=47.4s errors=0
[ZigZag/candidate] [ 325/499] elapsed=51.5s errors=0
[ZigZag/candidate] [ 350/499] elapsed=55.5s errors=0
[ZigZag/candidate] [ 375/499] elapsed=59.4s errors=0
[ZigZag/candidate] [ 400/499] elapsed=63.1s errors=0
[ZigZag/candidate] [ 425/499] elapsed=66.8s errors=0
[ZigZag/candidate] [ 450/499] elapsed=70.5s error

,threshold_fraction,is_primary_pre_registered_threshold,comparison_scope,automatic_threshold_selection_used,rows,eligible_events,candidate_events,positive_labels,negative_labels,upper_events,lower_events,vertical_events,event_return_sum,candidate_fraction_of_eligible,positive_meta_label_fraction,mean_event_return
0,0.10,False,train_only,False,658777,646618,110814,59629,51185,40403,19000,51411,2366.532236,0.171375,0.538100,0.021356
1,0.15,True,train_only,False,658777,646618,118464,63873,54591,45031,21599,51834,2573.748497,0.183206,0.539176,0.021726
2,0.20,False,train_only,False,658777,646618,102362,56044,46318,41021,19553,41788,2424.352152,0.158304,0.547508,0.023684


## 9. Freeze Stage 04 manifests

In [10]:
candidate_policy_manifest = {
    "stage": 4,
    "status": "frozen_after_integrity_checks",
    "primary_side": "long",
    "primary_threshold_fraction": PRIMARY_THRESHOLD,
    "threshold_selection_scope": "train_only",
    "automatic_threshold_selection": False,
    "sensitivity_threshold_fractions": SENSITIVITY_THRESHOLDS,
    "low_distance_column": candidate_policy["low_distance_column"],
    "high_distance_column": candidate_policy["high_distance_column"],
    "legacy_zigzag_columns_used_directly": False,
    "confirmation_gated_reconstruction": True,
    "meta_label_column": "meta_label",
    "meta_label_source": "frozen Stage 03 Triple-Barrier label",
    "unseen_test_used": False,
}

candidate_policy_path = (
    RESULT_PATHS["manifests"]
    / "04_candidate_filter_policy.json"
)
candidate_policy_path.write_text(
    json.dumps(
        candidate_policy_manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

run_manifest = {
    "stage": 4,
    "notebook": "04_feature_and_leakage_audit.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "frozen_universe_size": len(frozen_symbols),
    "approved_model_feature_count": len(approved_model_features),
    "approved_model_features": approved_model_features,
    "feature_approval_scope": "train_only",
    "unseen_test_used_for_feature_approval": False,
    "unseen_test_used_for_threshold_selection": False,
    "zigzag": {
        "depth": zigzag_config.depth,
        "deviation_percent": zigzag_config.deviation_percent,
        "legacy_new2_used_directly": False,
        "confirmation_gated_reconstruction": True,
        "prefix_checks": len(prefix_invariance_df),
        "prefix_mismatches": prefix_mismatches,
    },
    "candidate_filter": candidate_policy_manifest,
    "configuration_hash": stable_object_hash(
        {
            "columns": columns_config,
            "labeling": labeling_config,
            "validation": validation_config,
        }
    ),
}

run_manifest_path = (
    RESULT_PATHS["manifests"]
    / "04_feature_and_leakage_audit_manifest.json"
)
run_manifest_path.write_text(
    json.dumps(
        run_manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Candidate policy manifest:", candidate_policy_path)
print("Run manifest:", run_manifest_path)


Candidate policy manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\04_candidate_filter_policy.json
Run manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\04_feature_and_leakage_audit_manifest.json


## 10. Final integrity checks

In [11]:
approved_prohibited_audit = audit_prohibited_feature_names(
    approved_model_features,
    list(feature_policy["prohibited_feature_name_patterns"]),
)

legacy_zigzag_columns = set(
    feature_policy["legacy_zigzag_columns"]
)
candidate_output_files = sorted(
    CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv")
)

assert schema_errors_df.empty, (
    "Feature schema/quality errors exist. "
    "Review results/audits/04_feature_audit_errors.csv"
)
assert candidate_errors_df.empty, (
    "Candidate generation errors exist. "
    "Review results/audits/04_candidate_generation_errors.csv"
)
assert approved_model_features, "No model features were approved."
assert not approved_prohibited_audit["prohibited_name_hit"].any()
assert legacy_zigzag_columns.isdisjoint(approved_model_features)
assert set(feature_policy["context_only_price_features"]).isdisjoint(
    approved_model_features
)
assert prefix_mismatches == 0
assert (
    zigzag_state_audit_df["high_confirmation_after_event_rows"].sum()
    == 0
)
assert (
    zigzag_state_audit_df["low_confirmation_after_event_rows"].sum()
    == 0
)
assert threshold_comparison_df["comparison_scope"].eq(
    "train_only"
).all()
assert not threshold_comparison_df[
    "automatic_threshold_selection_used"
].any()
assert np.isclose(PRIMARY_THRESHOLD, 0.15)
assert len(candidate_output_files) == len(frozen_symbols)
assert int(
    candidate_audit_df["primary_candidate_events"].sum()
) > 0

primary_row = threshold_comparison_df.loc[
    np.isclose(
        threshold_comparison_df["threshold_fraction"],
        PRIMARY_THRESHOLD,
    )
].iloc[0]

print("Notebook 04 integrity checks: PASSED")
print("Frozen symbols audited:", len(frozen_symbols))
print("Approved model features:", len(approved_model_features))
print("Legacy ZigZag new_2 direct use: False")
print("Confirmation-gated ZigZag reconstruction: True")
print("Prefix-invariance mismatches:", prefix_mismatches)
print("Primary candidate threshold: 15%")
print("Threshold selection scope: train only")
print("Automatic threshold selection: False")
print("Primary candidate long events:", int(primary_row["candidate_events"]))
print(
    "Primary positive meta-label fraction:",
    round(float(primary_row["positive_meta_label_fraction"]), 6),
)
print("Unseen-test used in Stage 04 decisions: False")
print(
    "Next stage: Notebook 05 — Purged anchored walk-forward design "
    "on candidate long events."
)


Notebook 04 integrity checks: PASSED
Frozen symbols audited: 499
Approved model features: 25
Legacy ZigZag new_2 direct use: False
Confirmation-gated ZigZag reconstruction: True
Prefix-invariance mismatches: 0
Primary candidate threshold: 15%
Threshold selection scope: train only
Automatic threshold selection: False
Primary candidate long events: 118464
Primary positive meta-label fraction: 0.539176
Unseen-test used in Stage 04 decisions: False
Next stage: Notebook 05 — Purged anchored walk-forward design on candidate long events.


## Review before freezing Stage 04

Inspect:

- `results/audits/04_feature_approval_audit.csv`
- `results/audits/04_legacy_zigzag_source_logic_audit.csv`
- `results/audits/04_zigzag_prefix_invariance_audit.csv`
- `results/audits/04_confirmation_gated_zigzag_state_audit.csv`
- `results/audits/04_candidate_filter_threshold_comparison.csv`
- `results/audits/04_candidate_filter_by_year.csv`
- `results/audits/04_candidate_event_audit.csv`
- `results/audits/04_feature_audit_errors.csv`
- `results/audits/04_candidate_generation_errors.csv`
- `results/manifests/04_approved_model_features.csv`
- `results/manifests/04_candidate_filter_policy.json`
- `results/manifests/04_feature_and_leakage_audit_manifest.json`

Do not proceed to Notebook 05 until the final integrity checks pass and the
candidate-event density is reviewed.

## Recommended Git checkpoint after successful review

```bash
git add -A
git commit -m "feat: freeze leakage-audited features and confirmed ZigZag candidate events"
git push origin main

git tag -a milestone-04-feature-zigzag-audit -m "Freeze model features and confirmation-gated long-event filter"
git push origin milestone-04-feature-zigzag-audit
```
